# Ad Impression Data Analysis

This notebook analyzes the synthetic ad impression data generated by `src.data.ad_simulator`.

Checks covered:
- Overall CTR level
- CTR vs ad position (monotonic decrease expected)
- CTR vs cuisine match
- Bid amount distribution
- Train/val/test click-rate consistency

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

processed_dir = PROJECT_ROOT / "data" / "processed"
full_path = processed_dir / "ad_impressions.parquet"
schema_path = processed_dir / "ad_impressions_schema.json"

if not full_path.exists():
    raise FileNotFoundError("Run: .venv/bin/python3 -m src.data.ad_simulator first.")

ad_df = pd.read_parquet(full_path)
schema_doc = json.loads(schema_path.read_text()) if schema_path.exists() else {}

print(ad_df.shape)
ad_df.head()

## 1) Overall CTR

In [ ]:
overall_ctr = ad_df["click"].mean()

fig_ctr = px.bar(
    x=["overall"],
    y=[overall_ctr],
    labels={"x": "segment", "y": "CTR"},
    title="Overall Click-Through Rate",
)
fig_ctr.update_yaxes(tickformat=".2%")
fig_ctr.show()

display(Markdown(f"**Overall CTR:** `{overall_ctr*100:.2f}%`"))

if 0.04 <= overall_ctr <= 0.10:
    display(Markdown("✅ CTR is within the expected 4% to 10% range."))
else:
    display(Markdown("⚠️ CTR is outside the expected 4% to 10% range."))

## 2) CTR by Ad Position (Should Decrease)

In [ ]:
ctr_by_pos = (
    ad_df.groupby("ad_position", as_index=False)["click"]
    .mean()
    .sort_values("ad_position")
)

fig_pos = px.line(
    ctr_by_pos,
    x="ad_position",
    y="click",
    markers=True,
    title="CTR by Ad Position",
    labels={"click": "CTR"},
)
fig_pos.update_yaxes(tickformat=".2%")
fig_pos.show()

is_monotonic_decreasing = bool((ctr_by_pos["click"].diff().fillna(0) <= 0).all())

display(Markdown(f"**Monotonic decreasing CTR by position:** `{is_monotonic_decreasing}`"))
if not is_monotonic_decreasing:
    display(Markdown("⚠️ Observed minor non-monotonicity; re-run or increase sample size if needed."))

## 3) CTR by Cuisine Match

In [ ]:
ctr_by_match = (
    ad_df.groupby("cuisine_match", as_index=False)["click"].mean()
)
ctr_by_match["label"] = ctr_by_match["cuisine_match"].map({0: "No Match", 1: "Match"})

fig_match = px.bar(
    ctr_by_match,
    x="label",
    y="click",
    title="CTR by Cuisine Match",
    labels={"click": "CTR", "label": "Cuisine Match"},
)
fig_match.update_yaxes(tickformat=".2%")
fig_match.show()

display(Markdown(
    f"CTR(no match) = `{ctr_by_match.loc[ctr_by_match['cuisine_match']==0, 'click'].iloc[0]*100:.2f}%`, "
    f"CTR(match) = `{ctr_by_match.loc[ctr_by_match['cuisine_match']==1, 'click'].iloc[0]*100:.2f}%`"
))

## 4) Bid Amount Distribution

In [ ]:
fig_bid = px.histogram(
    ad_df,
    x="bid_amount",
    nbins=80,
    title="Bid Amount Distribution (Log-normal expected)",
)
fig_bid.show()

fig_bid_log = px.histogram(
    x=np.log1p(ad_df["bid_amount"]),
    nbins=80,
    title="log1p(Bid Amount) Distribution",
    labels={"x": "log1p(bid_amount)"},
)
fig_bid_log.show()

## 5) Train/Val/Test CTR Stability

In [ ]:
split_ctr = ad_df.groupby("split", as_index=False)["click"].mean()
fig_split = px.bar(
    split_ctr,
    x="split",
    y="click",
    title="Click Rate by Split",
    labels={"click": "CTR"},
)
fig_split.update_yaxes(tickformat=".2%")
fig_split.show()

max_gap = float(split_ctr["click"].max() - split_ctr["click"].min())
display(Markdown(f"**Max split CTR gap:** `{max_gap*100:.2f}%`"))
if max_gap <= 0.01:
    display(Markdown("✅ Train/val/test click rates are very similar (no obvious leakage)."))
else:
    display(Markdown("⚠️ Split CTR gap is larger than expected; inspect time drift."))

## 6) Ground-Truth Coefficients and Summary

In [ ]:
if schema_doc:
    coeffs = schema_doc.get("true_coefficients", {})
    display(Markdown("### True simulation coefficients"))
    display(pd.DataFrame([coeffs]).T.rename(columns={0: "value"}))

summary_md = f"""
### Findings
- Overall CTR: **{overall_ctr*100:.2f}%**
- CTR by position monotonic decreasing: **{is_monotonic_decreasing}**
- Split max CTR gap: **{max_gap*100:.2f}%**

Expected pattern checks:
- Position effect should be negative in aggregate CTR curves.
- Cuisine match should have higher CTR than no-match.
- Bid distribution should be right-skewed (log-normal-like).
"""

display(Markdown(summary_md))